# 학교 프로젝트를 위한 수업 코드 참고 노트북

이 노트북은 `3_LLM_지식추출` 수업에서 필요한 부품을 원본과 설명으로 묶은 개인 참고 자료다.
원본 수업 폴더가 없는 컴퓨터에서도 **여기 포함된 원본 코드와 입력/의존성 설명을 읽을 수 있다.**

- **수업 원본 발췌**는 Markdown 코드 블록이다. 사업보고서 전용 코드이므로 실행 셀로 복사하지 않는다.
- **학교 적용 예시**는 새로 작성한 독립적인 코드 셀이다. 앞에서부터 실행하면 표준 라이브러리만으로 작동한다.
- 예시의 문구·문서 ID·검수 상태는 전부 인공 fixture다. 공식 지침이나 실제 검수 결과가 아니다.
- 실행 셀은 파일·DB·네트워크를 읽거나 쓰지 않는다. Cypher는 문자열로 구성만 하며 실행하지 않는다.
- 공통 계약/스키마를 변경하지 않는다. 최종 구현은 프로젝트의 실제 승인 데이터와 공통 모듈을 사용한다.

읽는 순서: R01 승인/보류 → R02 원문 근거 → R03 표준 ID → R04 출처별 ID → R05 적재 집합
→ R06 MERGE 설계 → R07 재실행/실패 → R08 조회/근거 경로 → R09 공통 응답 → R10 검증.

각 원본 발췌는 함수 본문을 그대로 포함한다. 외부 전역변수·데이터는 각 절의 의존성 표에 설명하며,
원본을 그대로 실행하는 데 필요한 수업 데이터/API/DB를 이 패키지에 포함하지 않는다.
학교 적용 예시는 그 의존성을 인수와 인공 데이터로 바꿔 전체 흐름을 보여 준다.


## R00. 독립 예시 데이터와 실행 환경

원본 수업 함수의 전역변수에 의존하지 않기 위해 아래 사전·목록을 사용한다.
실제 프로젝트에서는 JSONL의 검수 결과와 승인된 사전을 읽어서 구성한다.
`APPROVED`, 페이지 및 인용 확인 값은 예시용 표시이지 실제 검증의 증명이 아니다.


In [1]:
from copy import deepcopy
import hashlib
import json
import re
import unicodedata

REF_DOC_A = 'example_school_manual_v1'
REF_DOC_B = 'example_school_guide_v1'
ref_documents = [
    {'document_id': REF_DOC_A, 'title': '인공 매뉴얼 A', 'issuer': '예시', 'version': 'v1', 'region': 'NATIONAL', 'source_url': None},
    {'document_id': REF_DOC_B, 'title': '인공 가이드 B', 'issuer': '예시', 'version': 'v1', 'region': 'GYEONGGI', 'source_url': None},
]
ref_chunks = [
    {'document_id': REF_DOC_A, 'chunk_id': f'{REF_DOC_A}:p0010:t01r01',
     'text': '인공 예시: 담당 역할은 예시이다. 담임교사: 유증상 학생을 발견하면 보호자에게 연락한다.',
     'pdf_page': 10, 'printed_page': 8, 'section': '인공 예시 > 발견', 'page_checked': True},
    {'document_id': REF_DOC_B, 'chunk_id': f'{REF_DOC_B}:p0020:t01r01',
     'text': '인공 예시: 담당 역할은 예시이다. 담임: 보호자 연락 후 확인 내용을 기록한다.',
     'pdf_page': 20, 'printed_page': 17, 'section': '인공 예시 > 발견', 'page_checked': True},
]
ref_role_dictionary = [
    {'role_id': 'HOMEROOM_TEACHER', 'name': '담임교사', 'aliases': ['담임', '담임(담당)교사']},
    {'role_id': 'HEALTH_TEACHER', 'name': '보건교사', 'aliases': ['보건(담당)교사']},
]
ref_concept_dictionary = [{'concept_id': 'CONTACT_GUARDIAN', 'name': '보호자 연락'}]
ref_candidates = [
    {'candidate_id': 'fixture_a', 'document_id': REF_DOC_A, 'source_chunk_id': ref_chunks[0]['chunk_id'],
     'source_order': 1, 'review_status': 'APPROVED', 'source_checked': True,
     'action_text': '유증상 학생 발견 시 보호자 연락', 'role_text': '담임교사', 'basis': 'EXAMPLE',
     'scope': 'COMMON', 'stage': 'RESPONSE_1', 'step_order': 1,
     'when_text': '유증상 학생을 발견하면', 'timing_text': None, 'modality': 'UNSPECIFIED',
     'concept_candidate': 'CONTACT_GUARDIAN', 'disease_ids': [],
     'evidence': '유증상 학생을 발견하면 보호자에게 연락한다.'},
    {'candidate_id': 'fixture_b', 'document_id': REF_DOC_B, 'source_chunk_id': ref_chunks[1]['chunk_id'],
     'source_order': 1, 'review_status': 'APPROVED', 'source_checked': True,
     'action_text': '보호자 연락', 'role_text': '담임', 'basis': 'EXAMPLE',
     'scope': 'COMMON', 'stage': 'RESPONSE_1', 'step_order': None,
     'when_text': None, 'timing_text': None, 'modality': 'UNSPECIFIED',
     'concept_candidate': 'CONTACT_GUARDIAN', 'disease_ids': [], 'evidence': '보호자 연락'},
]
print('인공 데이터 준비:', len(ref_chunks), '청크 /', len(ref_candidates), '후보')


인공 데이터 준비: 2 청크 / 2 후보


## R01. 모양 검사와 승인/보류 분리

수업 `shape_ok`는 Pydantic `Relation`과 `ValidationError`에 의존하고, 관계 스키마는 회사·품목·공급용이다.
`clean_rows`는 pandas DataFrame과 `mark_self`, `names`, `cut_tail`, `split_items`, `hold_reason`을 참조한다.
입력 열은 subject/object/relation/rcept_no 등, 출력은 원문 보존 열과 hold 이유를 추가한 표다.

가져올 원리는 **모양 검사, 원래 값 보존, 보류 이유 기록, 통과 집합 분리**다.
학교용 예시는 이 원리를 일반 사전/목록으로 구현한다. 원본의 회사 문자열 정리 규칙을 복제하지 않는다.


**수업 원본 발췌 — `Relation`**

출처: `40-5_구조화출력-스키마검증과-Neo4j-적재_배포용.ipynb`, 셀 11(Markdown 포함 1부터). 발췌 SHA-256: `b1271fec96409ff7c5c7059807cfcf4a40616d3b6f6a18f73c81f16f5092196c`.

```python
class Relation(BaseModel):
    subject: str = Field(min_length=1)
    relation: Literal["생산", "매입", "공급"]
    object: str = Field(min_length=1)
    item: str = ""
```


**수업 원본 발췌 — `shape_ok`**

출처: `40-5_구조화출력-스키마검증과-Neo4j-적재_배포용.ipynb`, 셀 16(Markdown 포함 1부터). 발췌 SHA-256: `864160c9a4e1cefb8491a0657c3fa3377b133a8983e08bb4eaf1f8c33787d5bd`.

```python
def shape_ok(r):
    """레코드 하나가 Relation 모양에 맞나"""
    try:                                             # (참고) 검증 결과를 참 · 거짓으로 받으려고 try 를 쓴다
        Relation.model_validate(r)
        return True
    except ValidationError:
        return False
```


**수업 원본 발췌 — `hold_reason`**

출처: `41-3_추출결과-정리-노이즈_배포용.ipynb`, 셀 27(Markdown 포함 1부터). 발췌 SHA-256: `d398e9da5a6d5e56eb875a4a747163de9a7cc7b2ce342da8c10dd9c9590d2f39`.

```python
def hold_reason(r):
    """관계 한 줄을 보류할 이유 - 문제없으면 빈 글자. 위에서부터 처음 걸린 것 하나"""
    if r["subject"].strip("- ") == "" or r["object"].strip("- ") == "":
        return "빈칸 · 기호"
    if r["subject"] == r["object"]:
        return "주어와 목적어가 같음"
    if r["relation"] in ("생산", "매입") and any(w in r["object"] for w in NOT_ITEM):
        return "품목이 아닌 말"                                      # 품목 자리에만 본다(회사 이름에는 안 씀)
    if len(MARK.findall(r["subject"])) >= 2 or len(MARK.findall(r["object"])) >= 2:
        return "회사 이름 둘이 붙음"                                 # ㈜A㈜B - 나누지 않고 사람이 본다
    return ""
```


**수업 원본 발췌 — `clean_rows`**

출처: `41-3_추출결과-정리-노이즈_배포용.ipynb`, 셀 41(Markdown 포함 1부터). 발췌 SHA-256: `d9a253be7ec1efbfc9d3610fe968f6d92c22a857dab6e847be4908668acea012`.

```python
def clean_rows(table, steps=("self", "tail", "split", "hold")):
    """관계 표 -> 정리한 표. 원래 이름은 subject_raw · object_raw 칸에 남기고, 보류는 hold 칸에 이유만 적는다(지우지 않음)
    steps: self 제출회사 표기 맞추기 / tail 꼬리 떼기 / split 품목 나누기 / hold 보류 이유 적기"""
    rows = []
    for r in table.to_dict("records"):
        r = dict(r, subject_raw=r["subject"], object_raw=r["object"])   # 원래 이름 보존
        if "self" in steps:                                            # 제출회사 이름의 다른 표기 -> @제출회사
            r["subject"] = mark_self(r["subject"], names[r["rcept_no"]])
            r["object"] = mark_self(r["object"], names[r["rcept_no"]])
        if "tail" in steps:
            r["subject"], r["object"] = cut_tail(r["subject"]), cut_tail(r["object"])
        objects = [r["object"]]
        if "split" in steps and r["relation"] in ("생산", "매입"):      # 품목 자리만 나눈다
            objects = split_items(r["object"]) or [r["object"]]
        for obj in objects:                                            # 나눈 품목마다 한 줄
            row = dict(r, object=obj)
            row["hold"] = hold_reason(row) if "hold" in steps else ""
            rows.append(row)
    return pd.DataFrame(rows, columns=list(table.columns) + ["subject_raw", "object_raw", "hold"])
```


**수업 원본 발췌 — `kept`**

출처: `41-3_추출결과-정리-노이즈_배포용.ipynb`, 셀 41(Markdown 포함 1부터). 발췌 SHA-256: `0678cc052301bd0d5962b6351c973684f970d55022647035275b63783da17723`.

```python
def kept(table):
    """보류하지 않은 줄만 - 채점 · 다음 단계에 쓰는 줄"""
    return table[table["hold"] == ""]
```


In [2]:
REF_SCOPES = {'COMMON', 'DISEASE_GROUP', 'NAMED_DISEASES', 'UNKNOWN'}
REF_STAGES = {'PREVENTION', 'RESPONSE_1', 'RESPONSE_2', 'RESPONSE_3', 'RECOVERY', 'ALL_STAGES', 'NATIONAL_CRISIS', 'UNSPECIFIED'}
REF_MODALITIES = {'MANDATORY', 'RECOMMENDED', 'DISCRETIONARY', 'UNSPECIFIED'}

def ref_shape_reasons(row):
    reasons = []
    for key in ['candidate_id', 'document_id', 'source_chunk_id', 'action_text', 'evidence']:
        if not isinstance(row.get(key), str) or not row[key].strip():
            reasons.append(f'필수 문자열 누락: {key}')
    for key, allowed in [('scope', REF_SCOPES), ('stage', REF_STAGES), ('modality', REF_MODALITIES)]:
        if row.get(key) not in allowed:
            reasons.append(f'허용값 위반: {key}')
    order = row.get('step_order')
    if order is not None and (type(order) is not int or order < 1):
        reasons.append('step_order 형식 위반')
    if row.get('role_text') and row.get('basis') not in {'EXPLICIT', 'EXAMPLE'}:
        reasons.append('역할 basis 누락')
    if row.get('review_status') != 'APPROVED' or row.get('source_checked') is not True:
        reasons.append('승인 또는 원문 검수 미완료')
    return reasons

def ref_partition(rows, checker):
    ready, held = [], []
    for raw in rows:
        row = deepcopy(raw)
        row['raw'] = deepcopy(raw)
        row['hold_reasons'] = checker(row)
        (held if row['hold_reasons'] else ready).append(row)
    return ready, held

ref_ready, ref_held = ref_partition(ref_candidates, ref_shape_reasons)
assert len(ref_ready) == 2 and not ref_held
print('형식/승인 검사 예시 통과')


형식/승인 검사 예시 통과


## R02. 실제 원문과 청크 소속 대조

`find_row(block, names, evidence)`는 `kind`, `rows`를 가진 수업 블록을 입력받아 유일한 근거 행 번호 또는 -1을 반환한다.
`check(r)`의 전역 의존성은 RANGE(관계 허용 목록), blocks_of(문서/블록 사전), block_all,
ends, COMPANY_MARK이다. 입력 r에는 subject/object/item/relation/row/evidence/rcept_no/block_id가 필요하다.
판정·이유의 튜플을 반환한다. 아래에 block_all과 ends도 함께 포함했다.

학교 적용에서는 청크 문서 소속, 실제 PDF 검수, 연속 인용과 해당 표 행을 확인한다.
**문자열 포함 검사는 의미 검증을 대신하지 않는다.** 표 제목·머리글·행이 불연속이면 원문 구성요소별 대조가 추가로 필요하다.


**수업 원본 발췌 — `find_row`**

출처: `40-7_개체관계-추출과-적재-근거검증_배포용.ipynb`, 셀 30(Markdown 포함 1부터). 발췌 SHA-256: `507e64089f63be39288dc658dab9c9ed0d7268a26a20c4cdd986c9caf242d86b`.

```python
def find_row(block, names, evidence=""):
    """표에서 근거 행 번호 - 이름이 다 든 행이 여럿이면 근거 글로 좁히고, 하나로 안 정해지면 -1(0 = 머리글은 빼고)"""
    if block is None or block["kind"] != "table":
        return -1
    rows = {i: " ".join(row) for i, row in enumerate(block["rows"][1:], start=1)}
    hits = [i for i, t in rows.items() if all(n in t for n in names)]       # 이름이 다 든 행
    parts = [p.strip() for p in str(evidence).split("|") if p.strip()]     # 근거 글 조각
    if len(hits) > 1 and parts:
        hits = [i for i in hits if all(p in rows[i] for p in parts)]       # 근거 글까지 든 행만
    if len(hits) == 1:
        return hits[0]
    return -1
```


**수업 원본 발췌 — `block_all`**

출처: `40-7_개체관계-추출과-적재-근거검증_배포용.ipynb`, 셀 31(Markdown 포함 1부터). 발췌 SHA-256: `3dbeacd21bb76f6db9f3992b4e9d04147eaffffb5d47150e9fd0604bee88d393`.

```python
def block_all(block):
    """블록의 글자 전부 - 표면 칸 글자를 이어 붙임"""
    if block["kind"] == "text":
        return block["text"]
    return " ".join(" ".join(row) for row in block["rows"])
```


**수업 원본 발췌 — `ends`**

출처: `40-7_개체관계-추출과-적재-근거검증_배포용.ipynb`, 셀 32(Markdown 포함 1부터). 발췌 SHA-256: `7c2ff14fd9c505f71504e02215a82ce10c28f9b3dab31f13194dd124418f8250`.

```python
def ends(r):
    """대조할 이름들 - 양 끝(@제출회사 빼고) + 품목"""
    items = [x for x in str(r["item"]).split(", ") if x]
    return [x for x in [r["subject"], r["object"]] + items if x and x != "@제출회사"]
```


**수업 원본 발췌 — `check`**

출처: `40-7_개체관계-추출과-적재-근거검증_배포용.ipynb`, 셀 34(Markdown 포함 1부터). 발췌 SHA-256: `befdaedd545f88eec9c8622d43315ebc3205641ebbae7bf6047666cbbc8a8ecc`.

```python
def check(r):
    """정답 없이 하는 검사 -> (판정, 이유). 위에서부터 처음 걸린 것 하나"""
    if r["subject"] == "" or r["object"] == "" or r["relation"] not in RANGE:
        return "위반", "빈 칸 · 허용 안 된 관계"
    if r["subject"] == r["object"]:
        return "위반", "주어와 목적어가 같음"                     # 품목을 item 칸에 잘못 넣은 경우가 많음
    block = blocks_of.get((r["rcept_no"], r["block_id"]))
    if block is None:
        return "위반", "근거 블록이 그 보고서에 없음"
    text = block_all(block)
    if not all(name in text for name in ends(r)):
        return "확인 필요", "양 끝 · 품목이 근거 블록에 없음"
    parts = [p.strip() for p in str(r["evidence"]).split("|") if p.strip()]   # 근거 글 조각들
    if len(parts) == 0 or not all(p in text for p in parts):
        return "확인 필요", "근거 글이 원문과 다름"
    if r["relation"] == "공급" and ("연결실체" in text or "계약" in text):
        return "확인 필요", "연결실체 · 계약 언급 - 원문 확인"
    if r["relation"] in ("생산", "매입") and COMPANY_MARK.search(r["object"]):
        return "확인 필요", "품목 자리에 법인 표기"
    if block["kind"] == "table" and r["row"] < 1:
        return "확인 불가", "표의 어느 행인지 하나로 못 정함"
    if block["kind"] == "table" and not all(p in " ".join(block["rows"][r["row"]]) for p in parts):
        return "확인 필요", "근거 글이 찾은 행과 다름"
    return "통과", ""
```


In [3]:
def ref_source_reasons(row, chunks_by_id):
    chunk = chunks_by_id.get(row['source_chunk_id'])
    if chunk is None:
        return ['청크 없음']
    reasons = []
    if chunk['document_id'] != row['document_id']:
        reasons.append('청크의 문서 소속 불일치')
    if chunk.get('page_checked') is not True:
        reasons.append('PDF 페이지 검수 대기')
    if not row['evidence'].strip() or row['evidence'] not in chunk['text']:
        reasons.append('연속 인용 불일치')
    return reasons

ref_chunks_by_id = {c['chunk_id']: c for c in ref_chunks}
assert all(not ref_source_reasons(r, ref_chunks_by_id) for r in ref_ready)
print('원문 소속/인용 검사 예시 통과; 의미·역할은 별도 검수 필요')


원문 소속/인용 검사 예시 통과; 의미·역할은 별도 검수 필요


## R03. 표준 사전 연결과 미연결 사유

수업 `rule_link(row)`는 mention을 가진 행, normalize, by_name(정규화 이름 → 회사 행 인덱스),
corps(DataFrame의 corp_code/상장 열)에 의존한다. 반환은 (코드 또는 빈 문자열, NIL 사유)다.
학교에서는 상장회사 우선 규칙을 제거하고 승인된 역할·질병·개념의 유일한 사전 연결만 채택한다.
유사한 이름을 찾는 것과 같은 역할/행동이라고 승인하는 것은 다르다.


**수업 원본 발췌 — `normalize`**

출처: `42-5_Entity-Linking-회사이름을-공시목록에-연결_배포용.ipynb`, 셀 11(Markdown 포함 1부터). 발췌 SHA-256: `d94eb1e084b3efe662fbdf8672f2f416cdfdd4463c02ae4dd9d2f6eec01e1234`.

```python
def normalize(name):
    """비교용 이름 - 전각 · 기호를 보통 글자로, 법인 표기 · 공백을 빼고, 영문은 대문자로"""
    name = unicodedata.normalize("NFKC", name)          # （주） -> (주), ㈜ -> (주)
    name = re.sub(r"주식회사|\(주\)|유한회사|\(유\)", "", name)     # '국도화학(주)' -> '국도화학'
    return name.replace(" ", "").upper()                   # 'LS MnM' -> 'LSMNM'
```


**수업 원본 발췌 — `rule_link`**

출처: `42-5_Entity-Linking-회사이름을-공시목록에-연결_배포용.ipynb`, 셀 23(Markdown 포함 1부터). 발췌 SHA-256: `c6fd1c3bea54370548e1644aa6f9199a179997326f8a3bc1f395663e4d1ba170`.

```python
def rule_link(row):
    """규칙으로 고르기 -> (고유번호 또는 '', NIL 이유)"""
    exact = list(by_name.get(normalize(row.mention), []))
    if len(exact) == 1:
        return corps.at[exact[0], "corp_code"], ""
    if len(exact) > 1:
        listed_ones = [i for i in exact if corps.at[i, "상장"]]
        return (corps.at[listed_ones[0], "corp_code"], "") if len(listed_ones) == 1 else ("", "같은 이름 여러 곳")
    return "", "목록에 없음"
```


In [4]:
def ref_compare_text(text):
    return re.sub(r'\s+', '', unicodedata.normalize('NFKC', text)).casefold()

def ref_resolve_role(mention, dictionary):
    target = ref_compare_text(mention)
    hits = {
        row['role_id'] for row in dictionary
        if target in {ref_compare_text(s) for s in [row['name'], *row['aliases']]}
    }
    if len(hits) == 1:
        return {'id': next(iter(hits)), 'status': 'RESOLVED', 'reason': ''}
    return {'id': None, 'status': 'AMBIGUOUS' if hits else 'UNRESOLVED',
            'reason': '여러 표준 역할' if hits else '사전에 없음'}

ref_normalized = []
for row in ref_ready:
    mapped = ref_resolve_role(row['role_text'], ref_role_dictionary)
    assert mapped['status'] == 'RESOLVED'
    assert row['concept_candidate'] in {x['concept_id'] for x in ref_concept_dictionary}
    ref_normalized.append({**deepcopy(row), 'role_id': mapped['id'], 'concept_id': row['concept_candidate']})
assert ref_resolve_role('담임', ref_role_dictionary)['id'] == 'HOMEROOM_TEACHER'
assert ref_resolve_role('아직 없는 역할', ref_role_dictionary)['id'] is None
print('역할 별칭 연결과 NIL 처리 확인')


역할 별칭 연결과 NIL 처리 확인


## R04. 출처별 행동 ID와 분할 기준

수업 `node_key(name, kind)`는 label/key/display-name의 튜플을 반환한다. @제출회사 처리에는
code_of, corp, target 등 수업 전역값을 참조한다. `OBS`는 원문 관측을 구별하는 칼럼 목록이었다.
학교 ID는 온톨로지 v3.1 형식과 source_manifest를 따르며 수업 회사 키를 그대로 쓰지 않는다.

아래 예시는 **고정된 fixture 집합**에서만 안정 정렬을 보여 준다. 실제 증분 처리에서는 매 실행
필터 결과를 새로 enumerate하지 말고 승인된 원문 위치·분할과 ID 매핑을 보존한다.
동일 조건/텍스트라도 문서가 다르면 별도 SchoolAction으로 유지한다.


**수업 원본 발췌 — `node_key`**

출처: `40-7_개체관계-추출과-적재-근거검증_배포용.ipynb`, 셀 43(Markdown 포함 1부터). 발췌 SHA-256: `62e566796767fc64de9d13fd2b2f104e12d28fec69e333747dd9ab6cbf389f47`.

```python
def node_key(name, kind):
    """(이름, 종류) -> (라벨, 키, 보여 줄 이름)"""
    if name == "@제출회사":
        return "Kg40Company", f"dart:{code_of[corp]}", corp
    if kind == "회사":
        return "Kg40Company", f"mention:{target}:{name}", name
    return "Kg40Item", f"{code_of[corp]}:{name}", name
```


In [5]:
def ref_assign_fixture_action_ids(rows, chunks_by_id):
    counters, mapped = {}, []
    ordered = sorted(rows, key=lambda r: (r['document_id'], chunks_by_id[r['source_chunk_id']]['pdf_page'], r['source_order'], r['candidate_id']))
    for row in ordered:
        page = chunks_by_id[row['source_chunk_id']]['pdf_page']
        key = (row['document_id'], page)
        counters[key] = counters.get(key, 0) + 1
        action_id = f"{row['document_id']}:p{page:04d}:a{counters[key]:02d}"
        mapped.append({**deepcopy(row), 'action_id': action_id})
    return mapped

ref_actions = ref_assign_fixture_action_ids(ref_normalized, ref_chunks_by_id)
assert ref_actions == ref_assign_fixture_action_ids(list(reversed(ref_normalized)), ref_chunks_by_id)
assert len({r['action_id'] for r in ref_actions}) == 2
print('출처별 ID:', [r['action_id'] for r in ref_actions])


출처별 ID: ['example_school_guide_v1:p0020:a01', 'example_school_manual_v1:p0010:a01']


## R05. 적재용 집합과 허용 관계 패턴

수업 `to_rows(df)`는 회사 코드(code_of), company_key, 관계 유형 매핑(REL_TYPE), 관측 키(OBS),
속성 변환(make_props)에 의존한다. 입력 DataFrame을 적재용 키·라벨·속성 사전 목록으로 바꾼다.
`check_sc40()`는 driver, Result.to_df와 ALLOWED를 사용해 DB의 관계 타입을 점검한다.

학교 변환에서는 노드 속성을 v3.1의 승인 목록으로 골라 만든다. 모델명·검수 상태·해시를
노드 속성에 임의 추가하지 않는다. 표준 사전 노드는 승인된 방식으로 준비하고 개인 적재에서 덮어쓰지 않는다.


**수업 원본 발췌 — `company_key`**

출처: `40-5_구조화출력-스키마검증과-Neo4j-적재_배포용.ipynb`, 셀 30(Markdown 포함 1부터). 발췌 SHA-256: `aad742a87ec728d20428222495cf147fa291d5a08fcf62ee94c6cf3a90151fb1`.

```python
def company_key(name, rcept_no):
    """보고서를 낸 회사 = 고유번호 키, 그 밖의 회사 = 보고서마다 따로인 임시 키"""
    if name in code_of:
        return f"dart:{code_of[name]}"
    return f"mention:{rcept_no}:{name}"
```


**수업 원본 발췌 — `make_props`**

출처: `40-5_구조화출력-스키마검증과-Neo4j-적재_배포용.ipynb`, 셀 30(Markdown 포함 1부터). 발췌 SHA-256: `35c183dffb76dfdb2ced207315b5aa7d601f0fad88ea8a21e0579417694ac221`.

```python
def make_props(r):
    """관계에 붙일 속성 - 금액 · 단위 · 비율은 값까지 통과한 관측에만"""
    props = {"rcept_no": r["rcept_no"], "block_id": r["block_id"], "row": int(r["row"]),
             "period": r["period"], "item": r["item"]}
    if r["판정"] == "통과":
        props.update({"value_status": "통과", "amount": r["amount"], "unit": r["unit"], "ratio": r["ratio"]})
    else:
        props["value_status"] = r["이유"]                          # 확인 불가 이유
    return props
```


**수업 원본 발췌 — `to_rows`**

출처: `40-5_구조화출력-스키마검증과-Neo4j-적재_배포용.ipynb`, 셀 31(Markdown 포함 1부터). 발췌 SHA-256: `c44492cc8de72ef8935dd9f5c8cec0715316ffe0bb7af60e24adafd47fe466ff`.

```python
def to_rows(df):
    """통과 · 확인 불가 레코드 -> 적재용 줄(키 · 라벨 · 관계 종류 · 관측 ID)"""
    rows = []
    for r in df.to_dict("records"):
        owner = code_of.get(r["corp_name"], r["corp_name"])
        if r["relation"] == "공급":
            o_label, o_key = "Sc40Company", company_key(r["object"], r["rcept_no"])   # 도착도 회사
        else:
            o_label, o_key = "Sc40Item", f"{owner}:{r['object']}"                      # 도착은 품목 - 회사마다 따로
        rows.append({"s_key": company_key(r["subject"], r["rcept_no"]), "s_name": r["subject"],
                     "o_label": o_label, "o_key": o_key, "o_name": r["object"],
                     "rel_type": REL_TYPE[r["relation"]], "props": make_props(r),
                     "obs_id": "|".join(str(r[c]) for c in OBS)})
    return rows
```


**수업 원본 발췌 — `check_sc40`**

출처: `40-5_구조화출력-스키마검증과-Neo4j-적재_배포용.ipynb`, 셀 36(Markdown 포함 1부터). 발췌 SHA-256: `4daa83c9935dba7e75e93e9edd159c392106a12cd28d8e0ab1107ef5aef5af51`.

```python
def check_sc40():
    """Sc40 노드에 붙은 모든 관계 -> 허용 목록에 없는 (출발, 관계, 도착)과 개수"""
    table = driver.execute_query("""
        MATCH (a)-[r]->(b) WHERE a:Sc40Company OR a:Sc40Item OR b:Sc40Company OR b:Sc40Item
        RETURN labels(a)[0] AS start, type(r) AS rel, labels(b)[0] AS end, count(*) AS n""", result_transformer_=Result.to_df)
    allowed = []
    for start, rel, end in zip(table["start"], table["rel"], table["end"]):
        allowed.append((start, rel, end) in ALLOWED)               # (출발, 관계, 도착)이 허용 목록에 있나
    table["허용"] = allowed
    return table
```


In [6]:
REF_ACTION_FIELDS = ['action_id', 'action_text', 'scope', 'stage', 'step_order', 'when_text', 'timing_text', 'modality']
REF_ALLOWED = {
    ('Document', 'HAS_CHUNK', 'Chunk'), ('Chunk', 'ABOUT', 'Disease'),
    ('SchoolAction', 'SUPPORTED_BY', 'Chunk'), ('SchoolAction', 'FOR_DISEASE', 'Disease'),
    ('SchoolAction', 'DONE_BY', 'SchoolRole'), ('SchoolAction', 'INSTANCE_OF', 'ActionConcept'),
}

def ref_build_bundle(rows, chunks):
    bundle = {'actions': [], 'chunks': [], 'edges': []}
    for chunk in chunks:
        bundle['chunks'].append({k: chunk[k] for k in ['chunk_id', 'text', 'pdf_page', 'printed_page', 'section']})
        bundle['edges'].append(('Document', chunk['document_id'], 'HAS_CHUNK', 'Chunk', chunk['chunk_id'], None))
    for row in rows:
        if row['scope'] != 'NAMED_DISEASES' and row['disease_ids']:
            raise ValueError('질병군/공통 범위에 특정 질병 적용 관계를 만들 수 없음')
        bundle['actions'].append({k: row[k] for k in REF_ACTION_FIELDS})
        base = ('SchoolAction', row['action_id'])
        bundle['edges'].extend([
            (*base, 'SUPPORTED_BY', 'Chunk', row['source_chunk_id'], None),
            (*base, 'DONE_BY', 'SchoolRole', row['role_id'], row['basis']),
            (*base, 'INSTANCE_OF', 'ActionConcept', row['concept_id'], None),
        ])
        for disease_id in row['disease_ids']:
            bundle['edges'].append((*base, 'FOR_DISEASE', 'Disease', disease_id, None))
    for start_label, _, rel, end_label, _, basis in bundle['edges']:
        if (start_label, rel, end_label) not in REF_ALLOWED:
            raise ValueError('허용하지 않은 관계')
        if rel == 'DONE_BY' and basis not in {'EXPLICIT', 'EXAMPLE'}:
            raise ValueError('basis 누락')
    return bundle

ref_bundle = ref_build_bundle(ref_actions, ref_chunks)
assert len(ref_bundle['actions']) == 2
assert 'review_status' not in ref_bundle['actions'][0]
assert len({edge[4] for edge in ref_bundle['edges'] if edge[2] == 'INSTANCE_OF'}) == 1
print('SchoolAction 2개를 공통 개념 1개로 연결한 적재 패키지 구성')


SchoolAction 2개를 공통 개념 1개로 연결한 적재 패키지 구성


## R06. UNWIND·MERGE의 실제 수업 패턴과 학교용 질의 템플릿

수업 44-1의 배치 질의 셀은 아래 원문과 같다. P는 실습 전용 라벨 접두사,
rows는 a/b/source 사전 목록이다. driver.execute_query(QUERY, rows=rows)로 목록을 전달했다.
원본 실행에는 수업의 Neo4j 연결이 필요하므로 여기서는 읽기용으로만 보인다.


**수업 원본 셀 전체** — `44-1_보고서-KG-Neo4j-적재-한번에_배포용.ipynb`, 셀 41. SHA-256 `64761e39d6ae44143572a43ba654fbc8a00b730e5fec0b7c11bc9c909f8ecb04`

```python
gold = pd.read_csv(DATA + "40-4_사업보고서_관계정답.csv", dtype={"rcept_no": str}).fillna("")
supply = gold[gold["relation"] == "공급"]
rows = []
for r, s, o in zip(supply["rcept_no"], supply["subject"], supply["object"]):
    a = names[r] if s == "@제출회사" else s           # @제출회사 -> 그 보고서의 회사 이름
    b = names[r] if o == "@제출회사" else o
    rows.append({"a": a, "b": b, "source": names[r]})
QUERY = f"""
UNWIND $rows AS row
MERGE (a:{P}GoldCompany {{name: row.a}})
MERGE (b:{P}GoldCompany {{name: row.b}})
MERGE (a)-[:{P}_GOLD_SUPPLIES {{source: row.source}}]->(b)"""
print(len(rows), "줄 /", rows[0])
```


학교 예시에서는 **ID만 MERGE 키**, 변경 가능한 속성은 SET으로 지정한다.
DONE_BY도 끝점으로 관계를 찾고 basis를 SET한다. basis 자체를 MERGE 키에 넣으면
예시/명시 판정 수정 시 관계가 두 개 생길 수 있다. 입력 끝점 누락은 MATCH에서 행을 소실시킬 수 있어
R08·R10의 기대/실제 키 대조가 필요하다. 다음 셀은 문자열만 준비한다.


In [7]:
ref_upsert_actions_cypher = """
UNWIND $rows AS row
MERGE (a:SchoolAction {action_id: row.action_id})
SET a.action_text = row.action_text, a.scope = row.scope, a.stage = row.stage,
    a.step_order = row.step_order, a.when_text = row.when_text,
    a.timing_text = row.timing_text, a.modality = row.modality
"""
ref_link_roles_cypher = """
UNWIND $rows AS row
MATCH (a:SchoolAction {action_id: row.action_id})
MATCH (role:SchoolRole {role_id: row.role_id})
MERGE (a)-[r:DONE_BY]->(role)
SET r.basis = row.basis
"""
# 위 문자열은 실행되지 않는다. 실제 쓰기 코드는 개인 index/school에 별도로 작성한다.
assert 'row.action_id' in ref_upsert_actions_cypher
print('학교용 질의 템플릿 준비; DB 호출 없음')


학교용 질의 템플릿 준비; DB 호출 없음


## R07. 해시와 실패 대상 재처리

수업 content_hash는 hashlib을 사용한다. 함께 발췌한 doc_key는 info DataFrame의
corp_code/fiscal_year를 사용해 보고서 묶음을 식별한다. run_all은 names와 ingest 함수를
참조하고 실패한 접수번호 목록을 반환한다.

학교 프로젝트는 입력 해시와 성공/실패 장부를 로컬 JSONL로 관리한다.
수업의 active/done/schema_version 속성을 승인 없이 공통 DB에 추가하지 않는다.


**수업 원본 발췌 — `doc_key`**

출처: `44-2_다시넣어도-안전하게-증분적재-버전-실패재처리_배포용.ipynb`, 셀 20(Markdown 포함 1부터). 발췌 SHA-256: `7c2707beacd98df645ef1d86d658fdb793f48219b0cb67647548678f0c81ae96`.

```python
def doc_key(rcept_no):
    """보고서 묶음 키 = 회사 고유번호 + 사업연도 (정정 공시는 같은 묶음, 새 사업연도는 새 묶음)"""
    return info.loc[rcept_no, "corp_code"] + "-" + info.loc[rcept_no, "fiscal_year"]
```


**수업 원본 발췌 — `content_hash`**

출처: `44-2_다시넣어도-안전하게-증분적재-버전-실패재처리_배포용.ipynb`, 셀 20(Markdown 포함 1부터). 발췌 SHA-256: `b3d9a411341b77ad305bd81b8f84d99bd8c198108d144b3b49630ff8eb03be4f`.

```python
def content_hash(text):
    """본문 해시 - 글자가 하나라도 다르면 다른 값 (앞 16자리만 씀)"""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]
```


**수업 원본 발췌 — `run_all`**

출처: `44-2_다시넣어도-안전하게-증분적재-버전-실패재처리_배포용.ipynb`, 셀 40(Markdown 포함 1부터). 발췌 SHA-256: `b2b675c7e707da2aa6c19dc1df8d0aba78fe507007a688a2a634a7a62181b0d2`.

```python
def run_all(targets, schema, schema_version):
    """여러 공시를 차례로 넣고 실패한 접수번호 목록을 반환 - 하나가 실패해도 다음으로 넘어간다"""
    failed = []
    for rcept_no in targets:
        try:                                             # (참고) 실패 처리가 이 날의 주제라 try 를 쓴다
            print(names[rcept_no], rcept_no, "->", ingest(rcept_no, schema, schema_version))
        except Exception as error:
            failed.append(rcept_no)
            print(names[rcept_no], rcept_no, "-> 실패:", type(error).__name__)
    print("실패", len(failed), "건")
    return failed
```


In [8]:
def ref_input_hash(value):
    body = json.dumps(value, ensure_ascii=False, sort_keys=True, allow_nan=False)
    return hashlib.sha256(body.encode('utf-8')).hexdigest()

def ref_run_units(units, worker):
    log = []
    for unit in units:
        try:
            worker(unit)
            log.append({'unit': unit, 'status': 'SUCCESS', 'error_type': None})
        except Exception as exc:
            log.append({'unit': unit, 'status': 'FAILED', 'error_type': type(exc).__name__})
    return log

def ref_fixture_worker(unit):
    if unit == 'bad':
        raise ValueError('인공 실패')

ref_run_log = ref_run_units(['a', 'bad', 'b'], ref_fixture_worker)
assert [r['status'] for r in ref_run_log] == ['SUCCESS', 'FAILED', 'SUCCESS']
assert ref_input_hash({'a': 1, 'b': 2}) == ref_input_hash({'b': 2, 'a': 1})
print('실패 격리와 JSON 입력 해시 확인')


실패 격리와 JSON 입력 해시 확인


## R08. 원문까지 조회하고 관계 키를 비교한다

수업 gold_keys는 driver와 P를 참조해 실제 DB 관계를 (a,b,출처) 집합으로 반환한다.
report_keys는 rcept_no/subject/relation/object 목록과 me/names에 의존해 내용 평가용 집합을 만든다.
**수업 report_keys는 블록·행을 무시**하므로 학교 프로젝트의 출처별 무결성 검증에 그대로 쓰면 안 된다.

학교에서는 action_id→chunk_id→document_id와 basis까지 실제로 조회하고 비교한다.
청크 검색기는 프로젝트 요구로 새로 작성할 부분이다. 이 폴더에 완성된 SchoolGraph 검색기가 있다고 가정하지 않는다.


**수업 원본 발췌 — `gold_keys`**

출처: `44-1_보고서-KG-Neo4j-적재-한번에_배포용.ipynb`, 셀 48(Markdown 포함 1부터). 발췌 SHA-256: `500319ecd9125c988ca069509dbc590bae63f63adb1576982e6d0a893c6665c5`.

```python
def gold_keys():
    """지금 그래프의 공급 관계를 (보내는 회사, 받는 회사, 출처 보고서) 집합으로"""
    found = driver.execute_query(f"MATCH (a:{P}GoldCompany)-[r:{P}_GOLD_SUPPLIES]->(b) RETURN a.name AS a, b.name AS b, r.source AS s")
    return {(x["a"], x["b"], x["s"]) for x in found.records}
```


**수업 원본 발췌 — `me`**

출처: `44-3_사업보고서22곳-적재와-채점_배포용.ipynb`, 셀 33(Markdown 포함 1부터). 발췌 SHA-256: `fe15c0420e23a46dca8b1d9d42223db9c5c6b26ef84a4a7760d201bf505beea2`.

```python
def me(rcept_no, name):
    """'당사' · 제출회사 이름(법인 표기 차이 포함)이면 @제출회사, 아니면 띄어쓰기만 맞춘 원문 이름"""
    bare = re.sub(r"㈜|\(주\)|주식회사|\s", "", str(name))   # '국도화학 ㈜' -> '국도화학'
    if bare in ("당사", names[rcept_no]):
        return "@제출회사"
    return re.sub(r"\s+", " ", str(name)).strip()                # 'LG  화학 ' -> 'LG 화학'
```


**수업 원본 발췌 — `report_keys`**

출처: `44-3_사업보고서22곳-적재와-채점_배포용.ipynb`, 셀 33(Markdown 포함 1부터). 발췌 SHA-256: `a046796fab0ee8fbd023fe568916dcc1d8fdd2954e0f00f1c819976cd49f95bb`.

```python
def report_keys(rows):
    """(복습) 관계 내용 점수의 키 (접수번호, 주어, 관계, 목적어) - 블록 · 행은 보지 않는다"""
    keys = set()
    for r in rows:
        keys.add((r["rcept_no"], me(r["rcept_no"], r["subject"]), r["relation"], me(r["rcept_no"], r["object"])))
    return keys
```


In [9]:
ref_evidence_query = """
MATCH (a:SchoolAction)-[:SUPPORTED_BY]->(c:Chunk)<-[:HAS_CHUNK]-(d:Document)
WHERE a.action_id IN $action_ids AND d.document_id IN $allowed_document_ids
OPTIONAL MATCH (a)-[r:DONE_BY]->(role:SchoolRole)
RETURN a.action_id AS action_id, d.document_id AS document_id,
       c.chunk_id AS chunk_id, c.text AS text, c.pdf_page AS pdf_page,
       c.printed_page AS printed_page, role.role_id AS role_id, r.basis AS basis
"""

def ref_edge_keys(bundle):
    # label/ID/관계/label/ID/basis를 포함한 키. 실제 DB 결과도 같은 형태로 바꾼다.
    return set(bundle['edges'])

def ref_compare_keys(expected, actual):
    return {'missing': sorted(expected - actual), 'extra': sorted(actual - expected)}

assert ref_compare_keys(ref_edge_keys(ref_bundle), ref_edge_keys(ref_bundle)) == {'missing': [], 'extra': []}
print('期待/実際のキー差分パターンを確認')


期待/実際のキー差分パターンを確認


## R09. 新規設計: Finding/CitationとSchoolGraphの外部境界

この節は**プロジェクトの契約 v1.2から新たに作る例**であり、上の会社コードの原文ではない。
State・検索・グラフ拡張は実装指示書で定義する。ここではDBレコードから契約応答を作る最小例を示す。
Fixtureの辞書をDB読み取り結果の代わりに使う。実際は検索・根拠検査後の結果だけを渡す。


In [10]:
def ref_empty_output(status, reason):
    return {'domain': 'school', 'status': status, 'findings': [], 'citations': [],
            'action_ids': [], 'warnings': [reason], 'answer': ''}

def ref_build_output(request, rows, chunks_by_id):
    allowed = set(request['allowed_document_ids'])
    findings, citations, used_actions = [], {}, set()
    for row in rows:
        chunk = chunks_by_id[row['source_chunk_id']]
        if chunk['document_id'] not in allowed:
            continue
        if ref_source_reasons(row, chunks_by_id):
            raise ValueError('근거 불일치')
        label = '담당 예시' if row['basis'] == 'EXAMPLE' else '원문의 담당'
        findings.append({'claim': f"[인공 예시] {row['role_text']}의 {label}: {row['action_text']}",
                         'chunk_ids': [chunk['chunk_id']], 'action_ids': [row['action_id']],
                         'disease_ids': row['disease_ids'], 'concept_ids': [row['concept_id']]})
        citations[chunk['chunk_id']] = {
            'document_id': chunk['document_id'], 'chunk_id': chunk['chunk_id'],
            'pdf_page': chunk['pdf_page'], 'printed_page': chunk['printed_page'],
            'excerpt': chunk['text'],
        }
        used_actions.add(row['action_id'])
    if not findings:
        return ref_empty_output('INSUFFICIENT_EVIDENCE', '허용된 범위에서 확인할 주장 없음')
    # 이 fixture에서는 좁은 작업 하나만 확인한다. 실제 ANSWERED/PARTIAL은 요청 충족도를 판단해야 한다.
    return {'domain': 'school', 'status': 'ANSWERED', 'findings': findings,
            'citations': list(citations.values()), 'action_ids': sorted(used_actions),
            'warnings': ['인공 데이터로 동작만 확인한 예시'], 'answer': ''}

ref_request = {'question': '인공 예시의 보호자 연락 담당은?', 'region': 'GYEONGGI',
               'allowed_document_ids': [REF_DOC_A, REF_DOC_B]}
ref_output = ref_build_output(ref_request, ref_actions, ref_chunks_by_id)
assert len(ref_output['findings']) == 2
print('공통 개념을 공유하되 출처별 Finding 2개 유지')


공통 개념을 공유하되 출처별 Finding 2개 유지


## R10. 새 구현에 필요한 검사와 예시의 한계

아래는 계약의 일부 조건과 원문 연결을 검사하는 **참고 예시**다. 공통 validate_query_output의 대체 구현이 아니다.
실제 구현은 팀 common 모듈의 전체 타입·상태·참조 검사 및 Neo4j 대조를 호출해야 한다.
원문 인용과 출처가 맞아도 주장의 의미·조건·강도·수행자는 사람/골든셋 검증이 필요하다.


In [11]:
def ref_validate_output_sources(output, request, chunks_by_id, actions_by_id):
    citations = {c['chunk_id']: c for c in output['citations']}
    if len(citations) != len(output['citations']):
        raise ValueError('중복 Citation')
    for citation in citations.values():
        chunk = chunks_by_id.get(citation['chunk_id'])
        if chunk is None or chunk['document_id'] != citation['document_id']:
            raise ValueError('원문 청크/문서 불일치')
        if citation['document_id'] not in request['allowed_document_ids']:
            raise ValueError('허용 문서 밖')
        if type(citation['pdf_page']) is not int or citation['pdf_page'] < 1 or citation['pdf_page'] != chunk['pdf_page']:
            raise ValueError('PDF 쪽수 불일치')
        if not citation['excerpt'].strip() or citation['excerpt'] not in chunk['text']:
            raise ValueError('인용 불일치')
    used = set()
    for finding in output['findings']:
        if not finding['chunk_ids'] or not set(finding['chunk_ids']) <= citations.keys():
            raise ValueError('Finding 근거 누락')
        for action_id in finding['action_ids']:
            action = actions_by_id.get(action_id)
            if action is None or action['source_chunk_id'] not in finding['chunk_ids']:
                raise ValueError('행동의 근거 연결 불일치')
        used.update(finding['action_ids'])
    if used != set(output['action_ids']):
        raise ValueError('행동 합집합 불일치')
    if output['status'] in {'INSUFFICIENT_EVIDENCE', 'ERROR'} and any(output[k] for k in ['findings', 'citations', 'action_ids']):
        raise ValueError('근거없음/오류 응답에 주장 또는 근거 포함')

ref_actions_by_id = {r['action_id']: r for r in ref_actions}
ref_validate_output_sources(ref_output, ref_request, ref_chunks_by_id, ref_actions_by_id)

def ref_expect_value_error(worker):
    try:
        worker()
    except ValueError:
        return
    raise AssertionError('예상한 검증 실패가 발생하지 않음')

# 검사: 허용 문서 없음, 존재하지 않는 근거, 잘못된 페이지, 모호한 사전, 미검수 후보.
ref_no_docs = {**ref_request, 'allowed_document_ids': []}
assert ref_build_output(ref_no_docs, ref_actions, ref_chunks_by_id)['status'] == 'INSUFFICIENT_EVIDENCE'
ref_broken = deepcopy(ref_output)
ref_broken['citations'][0]['chunk_id'] = 'missing'
ref_expect_value_error(lambda: ref_validate_output_sources(ref_broken, ref_request, ref_chunks_by_id, ref_actions_by_id))
ref_wrong_page = deepcopy(ref_output)
ref_wrong_page['citations'][0]['pdf_page'] = True
ref_expect_value_error(lambda: ref_validate_output_sources(ref_wrong_page, ref_request, ref_chunks_by_id, ref_actions_by_id))
ref_ambiguous = ref_role_dictionary + [{'role_id': 'OTHER_ROLE', 'name': '다른 역할', 'aliases': ['담임']}]
assert ref_resolve_role('담임', ref_ambiguous)['status'] == 'AMBIGUOUS'
assert ref_shape_reasons({**ref_candidates[0], 'review_status': 'PENDING'})
# 검사: 전체 문서에 한 개씩 있는 행동이 같은 개념이어도 하나로 병합되지 않음.
assert len({r['document_id'] for r in ref_actions}) == 2 and len({r['concept_id'] for r in ref_actions}) == 1
print('독립 예시 검사 완료: 실제 DB·LLM·공식 PDF 검증은 별도 수행')


독립 예시 검사 완료: 실제 DB·LLM·공식 PDF 검증은 별도 수행


## 다음 작업자가 실제 프로젝트에 적용하는 순서

같은 폴더의 `오늘_정규화_적재_질의_부품지시서.md`를 읽는다.
여기의 ref_ 함수는 입력/출력과 실패 처리를 보여 주는 예시이며 그대로 서비스에 import하지 않는다.
실제 코드에는 최신 승인 데이터, 공통 계약·사전·DB 의존성을 연결하고 실제 DB 조회 증빙을 남긴다.
원본 셀 실행, 초기화용 삭제, 공유 사전 임의 생성, 다른 담당자의 파일 수정은 이 패키지의 작업 범위가 아니다.
